# 交互式讲义说明

这是对应 Markdown 讲义的 notebook 版本。正文保持原有结构，末尾增加了真实公开数据实验、图像输出和解释问题。

- 环境：`research/.venv`；推荐从仓库根目录或 `research/lectures` 打开。
- 数据：默认从 `sources/external/data/` 本地读取；notebook 执行不会联网。
- 图像：执行后保存到 `research/lectures/figures/`，并在输出单元中显示。
- 研究边界：真实数据用于学习字段、时间对齐和统计直觉，不构成投资建议，也不代表可交易 alpha。

**原始讲义文件**：`04-data-pipeline.md`

In [ ]:
# Notebook setup: research/.venv + local public data
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the notebook runnable from research/lectures, research, or a parent workspace.
_here = Path.cwd().resolve()
_candidates = []
for _c in [_here, *_here.parents]:
    _candidates += [_c, _c / "research", _c / "lectures", _c / "research" / "lectures"]
for _candidate in _candidates:
    if (_candidate / "notebook_helpers.py").exists() and (_candidate.parent / "sources" / "external" / "data").exists():
        LECTURE_DIR = _candidate
        ROOT = _candidate.parent
        break
else:
    raise FileNotFoundError("请从 research、research/lectures 或包含 research 的工作区目录运行 notebook")
sys.path.insert(0, str(LECTURE_DIR))
from notebook_helpers import load_real_market_data, summary, annualized_sharpe, max_drawdown

FIG_DIR = LECTURE_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["figure.dpi"] = 110
print(f"ROOT={ROOT}")
print(f"Python={sys.executable}")
print("数据只从 sources/external/data 本地读取；要更新数据，请先保存新快照和来源说明。")

# 第四章：数据、偏差与量化研究管线（Data, Bias and the Research Pipeline）

> Quant research 的核心工作往往不是“训练一个更复杂的模型”，而是把一个模糊想法变成在当时信息集、真实交易规则和可复现实验中仍然成立的证据。本章给出一条从问题到上线候选的完整管线。

## 0. 学习目标

完成本章后，你应当能够：

- 把投资想法写成 hypothesis、universe、label、horizon、decision rule；
- 设计 data dictionary、point-in-time 数据集和可追溯的数据版本；
- 识别 survivorship bias、look-ahead bias、selection bias、data snooping、label leakage、backfill 和 stale price；
- 选择适合时间序列的 train/validation/test、walk-forward 和 purged split；
- 把研究 notebook 变成可重复运行的脚本、测试和研究 memo；
- 解释为什么一个漂亮回测可能无法交易，以及怎样逐层加上真实约束。

## 1. 从问题开始，而不是从数据开始

### 1.1 一个可检验的 hypothesis

不合格的问题是：“能不能用机器学习预测股票？”它没有定义资产、时间、信息集和成功标准。合格的研究问题至少包含：

> 在可交易股票宇宙中，若在时点 $t$ 观察到特征 X，是否能预测 $t+1$ 到 $t+h$ 的风险调整后收益，并在交易成本、容量和风险约束下改善一个预先指定的 baseline？

把它拆成：

- **universe（资产宇宙）**：哪些股票/期货/货币，按什么可获得规则进入；
- **feature information set（特征信息集）**：只使用在 $t$ 之前可见的字段；
- **label/target（标签）**：例如下一日简单收益、未来五日超额收益或方向；
- **horizon（预测期限）**：下一分钟、下一日、下周，和持仓/成交频率一致；
- **decision rule（决策规则）**：如何把预测转成仓位、排序或订单；
- **benchmark**：简单、透明、可复现的比较对象；
- **primary metric**：例如净 Sharpe、rank IC、最大回撤或成本后信息比率。

### 1.2 三个时间必须分开

每个字段都要区分：

1. **observation time**：经济事件发生或数据记录的时间；
2. **publication/availability time**：研究系统可以看到它的时间；
3. **execution time**：策略可以发出并成交订单的时间。

例如一家公司 3 月 31 日结束季度，4 月 20 日盘后公布财报，4 月 21 日才可能把公告反映到开盘交易。用 3 月 31 日作为特征时间会让回测偷看未来。

![报告期末、披露时间、可交易时间三个时间戳与泄漏窗口](figures/ch04_three_timestamps.png)

**图 4-1｜一个数字的三个时间戳。** 时间轴上依次是报告期末、披露时间和最早可交易时间；两者之间的红色区间就是泄漏窗口。在可交易时间之前使用该数字，回测就偷看了当时无人知道的信息。数据类型：概念示意图（水平距离为示意，不是真实财报日历）。

## 2. 数据字典和数据契约

### 2.1 Data dictionary

每个字段至少记录：

| 项目 | 示例 |
|---|---|
| field name | `adjusted_close` |
| economic meaning | 包含股息/分拆调整的价格 |
| unit/currency | USD/share |
| source and license | 数据供应商、授权范围 |
| frequency and timezone | daily, New York close |
| availability rule | close 后何时发布 |
| revision/backfill | 是否会修订历史 |
| missing-value rule | 不填、前值、删除或单独标记 |
| transformation | log、winsorize、rank、lag |
| owner/version | 数据版本、生成脚本和 checksum |

数据字典不是文档装饰。若两个人对“close”有不同理解，模型结果不可比较。

### 2.2 Data contract

在代码入口处定义可验证的契约：

- 日期索引单调、没有重复；
- 价格为正、成交量非负；
- 同一资产的币种和合约单位一致；
- 训练特征不含标签列或未来日期；
- 资产在当时确实属于 universe；
- 数据缺失率和异常值在阈值内。

让程序遇到违反契约时失败（fail fast），不要静默填补后继续产生一张漂亮图。

## 3. 常见偏差（Bias）

### 3.1 Survivorship bias（幸存者偏差）

只使用今天仍然存在的公司，会删除退市、破产和被并购的历史成员，使过去收益看起来更好。正确的 security master 应包含历史 constituent、进入/退出日期和退市收益处理。指数成分也必须按当时版本复建，不能使用今天下载的成分表回填过去。

![含退市成分的合成 universe 与幸存者口径的收益偏差](figures/ch04_survivorship.png)

**图 4-2｜幸存者偏差是一个水平位移。** 左图 8 个合成成分中有 3 个中途退市（红色虚线，退市前持续下跌）；右图比较两种口径的年化平均收益：只用今天仍存在的成分会给出明显更高的数字，因为失败路径被删掉了。数据类型：固定 seed 合成 universe。

### 3.2 Look-ahead bias（前视偏差）

常见来源：

- 用当日收盘价生成信号，又用当日收盘价成交；
- 用财报报告期末代替披露日期；
- 用最终修订后的宏观数据模拟当时决策；
- 用全样本均值、标准差或标准化参数处理早期数据；
- 在 rolling window 中包含了当前标签。

防护方法：所有决策都显式写成 `information available at t -> trade at t+1`，在代码中用 `shift(1)` 或成交延迟，并写一个单元测试构造“未来值改变但过去信号不能改变”的情况。

![信号与同日收益配对（泄漏）和与次日收益配对（正确）的 IC 对比](figures/ch04_lookahead_alignment.png)

**图 4-3｜对齐错了会制造假 IC。** 两幅散点使用同一份合成信号：左图把 signal_t 与 return_t 配对，IC 被系统性抬高，因为标签里包含信号当天的收益；右图把 signal_t 与 return_{t+1} 配对，才是“今天决策、明天承担收益”的真实关系。数据类型：固定 seed 合成 AR(1) 信号。

### 3.3 Selection bias（选择偏差）

如果只选择曾经成功的资产、市场或时间区间，样本不代表实际部署宇宙。研究报告要写出筛选规则是否在研究前固定，或是否根据结果选取。

### 3.4 Data snooping / multiple testing（数据窥探）

尝试 100 个信号、20 个窗口和 5 个成本假设，最后挑一个最好结果，相当于进行了大量隐含检验。必须保存 experiment log，记录参数、结果和失败尝试，用未触碰的测试区间或新数据做验证。

### 3.5 Label leakage（标签泄漏）

泄漏不只是在特征中直接放未来收益，也包括：

- 用整段样本计算标准化的均值和方差；
- 用未来成交量决定过去是否纳入样本；
- 用公告后修订的 sentiment 标签；
- 随机打乱时间序列，让相邻重叠标签同时出现在训练和测试。

### 3.6 Backfill、revision 和 stale price

供应商可能补齐过去缺失、修订宏观值、删除错误报价或用后验信息标注历史新闻。研究记录要写“as-of date”（截至日期）和版本。低频资产或停牌资产的价格长时间不动，会低估波动、夸大相关性并制造虚假 Sharpe。

## 4. 数据处理顺序

一个稳健的处理顺序是：

1. **Raw layer**：保存原始下载文件、来源、时间和 checksum，不直接覆盖；
2. **Normalization**：统一时区、列名、单位、币种和资产 ID；
3. **Corporate actions/contract mapping**：处理分红、分拆、换月、到期、退市；
4. **Point-in-time join**：按可见时间连接财务、新闻、宏观和行情；
5. **Feature construction**：只使用历史窗口，记录缺失和变换；
6. **Label construction**：先定义标签窗口，再确保它不进入特征；
7. **Quality checks**：重复、跳点、异常成交量、未来日期、缺失和覆盖率；
8. **Versioned dataset**：固定数据版本后才开始模型和回测。

处理顺序很重要：先把未来信息填进一个表，再“注意一点不要泄漏”通常来不及补救。

## 5. 时间序列切分与验证

### 5.1 Train/validation/test

普通机器学习随机切分假设样本可交换；市场数据有时间顺序，随机切分会让训练看到测试时期附近的信息。基本做法：

- train：用于拟合参数；
- validation：选择模型、特征和超参数；
- test：只在最后一次冻结后使用，用于最终样本外估计。

例如 2012–2018 train、2019–2020 validation、2021–2023 test。若研究期短，可以做 rolling/walk-forward，但必须保留一段真正未触碰的数据。

### 5.2 Walk-forward validation

在 walk-forward 中，每次用过去窗口训练，在下一个窗口验证，然后向前滚动：

```
train [2015, 2016] -> validate [2017 Q1]
train [2015, 2016, 2017 Q1] -> validate [2017 Q2]
...
```

这模拟模型逐步获得新信息的过程，能观察性能是否只出现在某一段 regime。若使用固定长度训练窗口，可适应关系变化但减少样本；扩展窗口更稳定但可能包含过时关系。

### 5.3 Purged 和 embargo split

如果标签是未来五日收益，相邻样本的标签会重叠。训练集最后一天的标签可能包含验证集第一天的价格，形成隐性泄漏。**Purging** 删除边界上会重叠的训练样本；**embargo** 在训练和验证之间留出一段间隔，防止信息通过持仓或标签传播。

![单次切分、walk-forward 与 purged + embargo 三种时间切分](figures/ch04_time_splits.png)

**图 4-4｜三种切分，一条规则：不打乱时间。** 第一行是最简单的训练/验证/测试切分；第二行是 walk-forward，训练窗滚动、测试窗跟随；第三行在边界加入 purge（删除标签跨边界的样本）与 embargo（安全间隔）。三者都保持时间顺序。数据类型：概念示意图。

## 6. 回测不是一张收益曲线

### 6.1 Signal、position、order、fill

把以下对象分开：

1. **signal**：模型/规则输出的预测、分数或方向；
2. **target position**：根据风险预算和约束想持有的仓位；
3. **order**：为了改变仓位而发送的订单；
4. **fill**：订单实际成交的数量和价格；
5. **portfolio/PnL**：成交后持仓、现金和损益。

简单项目可以用 `position.shift(1) * return` 近似，但必须在报告中说明它假设了什么。短周期或高换手研究需要模拟 bid/ask、部分成交、延迟、市场冲击、撤单和限额。

![信号、下单、成交的时间链与收益归属区间](figures/ch04_signal_to_fill.png)

**图 4-5｜哪一段收益属于这个信号。** 信号在 t 日收盘后用 ≤ t 的数据计算，订单在 t+1 开盘成交，因此 t+1 到 t+2 的收益才属于这个仓位。红色标注是常见错误：按 t 日收盘价成交，等于假设能在信号产生之前成交。数据类型：概念示意图（时刻为示意）。

### 6.2 回测中的顺序

每个时间步应明确：

```
读取 t 以前的数据
-> 计算信号
-> 应用信号延迟
-> 生成目标仓位
-> 与旧仓位比较得到订单
-> 按可成交价格和成本成交
-> 更新现金、持仓、PnL 和风险
```

如果先用未来收盘计算仓位，再乘同一日收益，结果会系统性偏高。建议在代码中保留 `signal_time`、`decision_time`、`execution_time` 三列，而不是只保留一个日期。

## 7. 研究报告的证据层级

一个从低到高的证据路径：

1. **sanity check**：信号方向和单位正确，图表没有明显错误；
2. **in-sample fit**：只说明规则能解释历史；
3. **out-of-sample**：冻结规则后在未见数据评估；
4. **cost/turnover stress**：不同成本和执行延迟下仍有空间；
5. **subperiod/regime**：不同年份、波动和市场状态；
6. **cross-asset/market**：换资产和市场后是否有同一经济机制；
7. **paper/live shadow**：不下真钱地记录实时信号与可成交价格；
8. **small production**：有限风险预算下监控漂移、异常和容量。

每一层发现问题都应回到假设，而不是继续调参掩盖问题。

## 8. Reproducibility（可复现）

至少保存：

- 代码版本（git commit）；
- Python/uv 环境和依赖锁定；
- 数据来源、版本、下载日期和 hash；
- 配置文件（资产、窗口、成本、随机种子）；
- 原始结果、图表和研究 memo；
- 实验日志和失败结果。

研究脚本要能从干净环境运行，不能依赖 notebook 中“之前执行过的某个 cell”。项目 A/B 提供了 `src.run` 和无 pytest 依赖的测试入口，目的就是练习这个习惯。

## 9. 研究 memo 模板

一页 memo 至少包括：

1. **Question**：研究什么，为什么可能存在；
2. **Data**：资产、字段、时间范围、point-in-time 规则；
3. **Method**：特征、标签、模型、持仓和执行；
4. **Baselines**：naive、equal-weight、lagged、random 或简单因子；
5. **Results**：gross/net、Sharpe、drawdown、turnover、cost、IC、子时期；
6. **Robustness**：参数、样本、成本、延迟、资产替换；
7. **Failure modes**：泄漏、容量、流动性、结构变化；
8. **Decision**：继续研究、缩小、停止或需要新数据。

研究结论必须包含证据和限制，避免只写“模型表现不错”。

## 10. 常见误解

1. **“清理数据越多越好。”** 过度填补和删异常可能把真实风险清掉；每个处理都要有经济理由。
2. **“随机 train/test 也能用在价格预测。”** 时间顺序和标签重叠让随机切分严重乐观。
3. **“回测只要扣手续费。”** spread、冲击、融资、借券、延迟和容量同样重要。
4. **“测试集可以不断查看。”** 一旦用测试集选模型，它就不再是测试集。
5. **“一个 notebook 产生结果就是可复现。”** 环境、数据版本、随机种子和执行顺序都需记录。

## 11. 手工练习：审计一个虚假回测

下面的策略规则是：“每天收盘后计算当天收益排名，买入排名最高的资产，并按当天收盘价成交。”请找出至少六个问题，并提出修复：

- 当天收益在收盘后才完整知道；
- 当天收盘价已经结束，无法按同一收盘价成交；
- 排名宇宙是否包含当日之后才知道的成分；
- 是否使用复权价格而没有处理实际现金流；
- 是否忽略 spread、冲击和换手；
- 资产停牌/缺失如何处理；
- 目标仓位何时生效；
- benchmark 和成本敏感性是否存在。

把修复写成时间顺序：`t close -> signal -> t+1 open/mid execution -> t+1 return`，并说明哪一列需要 shift。

## 补充：版本化数据与研究可追溯性

### Point-in-time 与 vintage 数据

宏观数据和财报数据可能在发布日期之后被修订。研究时要区分：`observation_period`（数据描述的期间）、`release_time`（首次公开时间）、`as_of_time`（研究者实际能拿到的版本）和 `revision_time`（后续修订时间）。如果用今天下载的最终 GDP 或 restated 财报回测十年前策略，模型得到的是“修订后的历史”，不是当时可交易的信息。

可审计的表至少保留 `source_id`, `version`, `published_at`, `effective_from`, `retrieved_at`, `content_hash`。同一文档更新时新增版本，不覆盖旧行；回测 join 的条件应是 `published_at <= decision_time` 且选择当时最后可见版本。

### 公司行动与价格字段

拆股、分红、退市、停牌和 ticker change 会改变价格序列。`raw_close` 适合模拟当时可成交价格，`adjusted_close` 适合某些长期收益统计，但不能把调整因子倒灌到历史订单成交。真实项目需要独立的 corporate-actions 表，并在 data contract 中写明因子生效日和现金流处理。

### 数据质量的三层测试

1. **schema tests**：字段、类型、主键和单位是否稳定；
2. **temporal tests**：发布时间、有效期、交易日和时区是否满足 as-of 约束；
3. **economic tests**：收益范围、价格非负、分红/拆股跳变、成交量和流动性是否合理。

只有三层都通过，数据才进入 feature store 或回测。把检查写成测试比在 notebook 里手工看一眼更可靠。

## 12. 本章小结

研究管线从问题定义开始，经过数据字典、point-in-time 对齐、特征标签构造、时间切分、回测、成本压力测试和复现，最后才得到一个候选结论。偏差不是附录问题，而是决定回测能否代表真实决策环境的核心。

最有价值的研究代码常常是能明确回答“这条信息在当时能不能看到”“这笔订单何时成交”“换一种合理成本后是否还成立”的代码。

## 13. 自测与面试表达

1. 列出 observation time、availability time、execution time 的区别，并用财报举例。
2. 什么是 survivorship bias？为什么股票指数回测特别容易遇到？
3. 未来五日收益作为标签时，为什么需要 purge/embargo？
4. 画出 signal、position、order、fill、PnL 的关系。
5. 用 90 秒描述你会怎样把一个 notebook 变成可复现研究项目。

## Reference 与进阶了解

- Marcos López de Prado, **Advances in Financial Machine Learning**（时间切分、purging、embargo、标签和研究偏差）。
- Michael Halls-Moore, **QuantStart** 的免费系列，涵盖 backtesting、event-driven architecture 和数据处理：<https://www.quantstart.com/articles/>。
- QuantEcon, **Data and Empirical Methods**：<https://quantecon.org/lectures/>。
- SEC EDGAR，观察披露时间与原始文件：<https://www.sec.gov/edgar/search/>。
- FRED，宏观数据的发布日期和修订说明：<https://fred.stlouisfed.org/>。
- WorldQuant BRAIN Research Consultant / data 说明（需遵守其平台规则）：<https://platform.worldquantbrain.com/>。
- pandas time series、scikit-learn model selection 文档，作为代码实现参考：<https://pandas.pydata.org/docs/>、<https://scikit-learn.org/stable/model_selection.html>。

# 可运行实验：同一条宏观观察，publication time 如何改变特征

真实数据里“观察期”和“发布日期”可能不同。下面用真实 VIX 日序列构造一个教学事件表：每个月的观测值只在当月第 5 个工作日发布。然后对 S&P 500 做 `merge_asof`，展示合法 as-of 特征何时出现；同时构造一个使用整月均值的 naive 特征，直观看到它把未来信息带回月初。

In [ ]:
market = load_real_market_data(ROOT).loc["2018":"2025"].copy()
monthly = market["vix"].resample("MS").mean().dropna().to_frame("vix_month_mean")
monthly["published_at"] = monthly.index + pd.offsets.BDay(5)
monthly = monthly.reset_index(names="observation_month").sort_values("published_at")
prices = market[["spx_close"]].reset_index().sort_values("date")
legal = pd.merge_asof(prices, monthly[["published_at", "vix_month_mean"]], left_on="date", right_on="published_at", direction="backward")
# This intentionally leaky comparison maps the final full-month mean to every day in that month.
naive = prices.copy()
naive["observation_month"] = naive["date"].values.astype("datetime64[M]")
naive = naive.merge(monthly[["observation_month", "vix_month_mean"]], on="observation_month", how="left")

first_legal = legal.loc[legal["vix_month_mean"].notna(), "date"].iloc[0]
first_naive = naive.loc[naive["vix_month_mean"].notna(), "date"].iloc[0]
print("first legal feature date:", first_legal.date(), "| naive date:", first_naive.date())
print("naive observations available earlier by days:", (first_legal - first_naive).days)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(legal["date"], legal["vix_month_mean"], label="as-of published feature", lw=2)
ax.plot(naive["date"], naive["vix_month_mean"], label="naive full-month mean (leaky)", alpha=.55)
ax.axvline(first_legal, color="black", ls="--", lw=1, label="first legal date")
ax.set_ylabel("VIX monthly mean")
ax.set_xlabel("date")
ax.legend()
ax.set_title("Publication time is part of the feature definition")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_asof_pipeline.png", bbox_inches="tight")
plt.show()

### 观察和面试提示

- `observation_month` 描述数据属于哪个期间；`published_at` 决定何时可以用。两者不能互换。
- 如果原始供应商后来修订了月均值，还需要保存 `retrieved_at`、版本和 hash；本实验只演示第一层 as-of 约束。
- 练习：把 `published_at` 延迟 2 个工作日，再观察合法特征覆盖率和可交易样本量。